Elvato Development Documentation
===============================
Headless Ecommerce built-on Typescript/ React

Medusa development Admin credentials:
user: admin@elvato.com
pass: supersecret

CJ-Dropship User Credentials:
user: elvaolighting@gmail.com
pass: Msi58900!



# Development Structure Overview

Architecture Overview

<u>Description:</u>
We have our backend Medusa.js application. Hosted on Railway for a backend API hosting URLs and Admin Panel, with postgresql database setup on Neon, and Redis cache/ Upstash (redis cache). We have a live Convex URL, which uses Bunny.net for image CDN. Our storefront is hosted on Vercel. 

<u>Services:</u> <br>
Medusa.js <br>
Railway <br>
Redis/ Upstash <br>
Neon <br>
Convex <br>
Bunny <br>
Vercel <br>
Stripe <br>
 






<u>Structure</u>:
Storefront |
Database |
Admin |
Logic 

**Development**
1. Storefront <br> 
localhost:8000 <br>
1.1. Procurement Portal included as storefront page <br> 
localhost:8000/shipping [in-progress] <br>
User interaction interface <br>
Host: Vercel

2. Catelogue Manager <br>
localhost:8008 <br>
Wholesale API and UI for managing products, categories, collections, orders, customers, and inventory <br>
Host: Vercel

3. Admin <br>
https://medusa-backend-production-d681.up.railway.app/app <br>
localhost:9000/admin <br>
npx medusa develop <br>
React/ Vite5 UI served BY backend which uses Admin API from backend <br>
Host: Vercel

4. Logic <br>
https://medusa-backend-production-d681.up.railway.app <br>
localhost:9000 <br>
Provides all Store API and Admin API endpoints <br>
Host: Render

1. Database Postgresql <br>
ep-floral-wildflower-aiom3gle <br>
localhost:5432 <br>
Stores all ecommerce data <br>
Host: Neon

# Competitive Analysis & Research

-FinesseDecor: https://finessedecor.com/collections/chandeliers
traffic: 4,141
bounce-rate: 43.97%
sales: n/a
rating: sucks 
Modern Lighting focus, interesing curation of likeminded products. 

-LightsCanada: https://lightscanada.ca/
traffic: 87,706/ month
bounce-rate: 36.91%
sales: n/a 
rating: pretty good
Large selection of lighting with a variety of collections and style.

-Lumens: https://www.lumens.com/
traffic: 1.451M
bounce-rate: 52.14%
sales: $14.98MM/ month 
rating: excellent
A top online lighting and modern furniture retailer.

-2modern: https://www.2modern.com/
traffic: 459,150
bounce-rate: 48.87%
sales: $1.43MM/ month
rating: pretty good
An e‑commerce platform specializing in contemporary design furniture and lighting. 

-ShadesofLight: https://www.shadesoflight.com/
traffic: 516,465
bounce-rate: 42.75%
sales: $2.95MM/ month
rating: pretty good
A national catalog/online lighting store.

-YLighting: https://www.ylighting.com/ (dead/ Wayfair now)
traffic: 1,583
bounce-rate: 39.96%
sales: n/a
rating: good
A long‑time leader in contemporary lighting (now a Wayfair brand). 

-MontrealLighting: https://www.montreallighting.com/
traffic: 38,843
bounce-rate: 30.28%
sales: n/a
rating: fair
A Canadian online lighting retailer with a wide selection of products.



# CJ-API Docs

### General Overview

CJ Dropshipping API powers the **Inventory** tab in the Catalogue Manager, enabling:
- Product discovery and search across CJ's catalog
- Filtering by lighting-specific categories
- Importing products to local database (TODO)

### Integration Architecture

```
┌─────────────────┐      ┌──────────────────┐      ┌─────────────────┐
│  Catalogue UI   │ ──▶  │  Next.js API     │ ──▶  │  CJ API v2.0    │
│  (Inventory)    │      │  /api/cj/*       │      │  developers.cj  │
└─────────────────┘      └──────────────────┘      └─────────────────┘
```

### Our API Routes

| Route | Method | Purpose |
|-------|--------|---------|
| `/api/cj/auth` | GET | Check auth status |
| `/api/cj/auth` | POST | Force token refresh |
| `/api/cj/products` | GET | Search products with filters |
| `/api/cj/categories` | GET | Get category tree (cached 1hr) |

### Authentication Flow

| Step | Action | Token |
|------|--------|-------|
| 1 | Request with `CJ_API_KEY` | → Access Token (15 days) |
| 2 | Cache token server-side | — |
| 3 | Auto-refresh when expired | Refresh Token (180 days) |

### Search Parameters We Use

| Parameter | Type | Description |
|-----------|------|-------------|
| `keyWord` | string | Search term (e.g., "chandelier") |
| `categoryId` | string | 3rd-level category UUID |
| `page` / `size` | int | Pagination (max 100/page) |
| `orderBy` | int | 0=match, 1=popularity, 2=price, 3=date |
| `sort` | string | `asc` or `desc` |
| `startSellPrice` / `endSellPrice` | float | Price range (USD) |

### Lighting Presets

| Preset | Keywords |
|--------|----------|
| All Lighting | lamp, light, LED |
| Pendant & Chandeliers | pendant, chandelier, hanging |
| Table & Desk Lamps | table lamp, desk lamp |
| Wall Lights | wall light, sconce |
| Smart Lighting | smart, wifi, RGB |
| Decorative | fairy, string, neon |
| Outdoor | garden, solar, pathway |

### Response Structure

```
CJ API Response:
├── code: 200
├── data
│   ├── pageNumber, pageSize, totalRecords
│   └── content[]
│       └── [0]
│           ├── list[] ← Products array
│           ├── relatedCategoryList[]
│           └── keyWord
```

### Environment Variables

| Variable | Location | Required |
|----------|----------|----------|
| `CJ_API_KEY` | `/catalogue/.env.local` | ✓ |

## Rate Limits

### Authentication Limits
| Endpoint | Limit |
|----------|-------|
| `getAccessToken` | **Once per 5 minutes** |
| `refreshAccessToken` | 5 times per minute |

### Token Lifecycle
| Token Type | Validity |
|------------|----------|
| Access Token | 15 days |
| Refresh Token | 180 days |

### General Request Limits
| Limit Type | Rate |
|------------|------|
| Per IP address | 10 requests/second |
| Non-login interfaces | 30 requests/second |

### User Level Limits (Elvato = Level 3 Prime)
| User Level | Requests/Second |
|------------|-----------------|
| Free / Level 0-1 | 1 req/sec |
| Plus / Level 2 | 2 req/sec |
| **Prime / Level 3** | **4 req/sec** ✓ |
| Advanced / Level 4-5 | 6 req/sec |

### Product List API Specific
- Free/V1 users: Limited to 1,000 requests/day
- 1 IP → max 3 user accounts

---

## Key Endpoints

### Authentication
- `POST /api2.0/v1/authentication/getAccessToken` - Get new token
- `POST /api2.0/v1/authentication/refreshAccessToken` - Refresh token

### Products
- `GET /api2.0/v1/product/listV2` - Search products (Elasticsearch)
- `GET /api2.0/v1/product/query` - Get product details
- `GET /api2.0/v1/product/getCategory` - Get category tree

### Required Headers
```
Content-Type: application/json
CJ-Access-Token: <your-access-token>
```

---

## Implementation Notes

**Token Caching Issue (Development)**
- In dev mode, hot-reloads clear the in-memory token cache
- Each restart requires waiting 5 min for new token
- Solution: Don't edit files while testing CJ API

**API Key Location**
- Dashboard: https://www.cjdropshipping.com/myCJ.html#/apikey
- Store in: `/catalogue/.env.local` as `CJ_API_KEY`

# Git/ Github Editor Cmd

## Creating GitHub Issues from Terminal

### Prerequisites
1. Install GitHub CLI: `brew install gh`
2. Authenticate: `gh auth login`

### Basic Commands

| Command | Description |
|---------|-------------|
| `gh issue create` | Interactive issue creation |
| `gh issue list` | List open issues |
| `gh issue view <number>` | View specific issue |
| `gh issue close <number>` | Close an issue |

### Create Issue (Interactive)
```bash
gh issue create
```
Prompts for: title, body, labels, assignees, project

### Create Issue (One-liner)
```bash
gh issue create --title "Bug: Fix header alignment" --body "Description here"
```

### With Labels and Assignees
```bash
gh issue create \
  --title "Feature: Add dark mode" \
  --body "Implement dark mode toggle in settings" \
  --label "enhancement" \
  --assignee "@me"
```

### From a File
```bash
gh issue create --title "New Feature" --body-file ./issue-description.md
```

### Common Options

| Flag | Description |
|------|-------------|
| `--title`, `-t` | Issue title |
| `--body`, `-b` | Issue body/description |
| `--body-file`, `-F` | Read body from file |
| `--label`, `-l` | Add labels (can repeat) |
| `--assignee`, `-a` | Assign users |
| `--project`, `-p` | Add to project |
| `--milestone`, `-m` | Add to milestone |
| `--web`, `-w` | Open in browser to finish |

### Examples for Elvato

```bash
# Bug report
gh issue create -t "Bug: Cart not updating" -b "Steps to reproduce..." -l "bug"

# Feature request  
gh issue create -t "Feature: Wishlist functionality" -l "enhancement" -a "@me"

# Quick issue, finish in browser
gh issue create -t "TODO: Refactor checkout flow" --web
```

# Convex =/= Medusa.Ja

## Product Schema Comparison: CJ Dropshipping ↔ Medusa PostgreSQL

### Convex `cjMyProducts` Table Schema
✓ (checkmark) = Required field - must have a value   
○ (circle) = Optional field - can be null/undefined

| Column | Type | Required | Description |
|--------|------|----------|-------------|
| `_id` | ID | ✓ | Convex auto-generated ID |
| `_creationTime` | number | ✓ | Convex auto-generated timestamp |
| `cjProductId` | string | ✓ | CJ's unique product identifier |
| `sku` | string | ✓ | Product SKU code |
| `nameEn` | string | ✓ | English product name |
| `productNames` | string[] | ✓ | All product names from CJ |
| `bigImage` | string | ✓ | Main image URL |
| `price` | number | ✓ | Price in USD (totalPrice from CJ) |
| `productType` | number | ✓ | CJ product type code |
| `listedShopNum` | string | ○ | Number of shops listed on |
| `cjCreatedAt` | string | ✓ | When added to CJ "My Products" |
| `syncedAt` | number | ✓ | When synced to Convex (Unix timestamp) |
| `updatedAt` | number | ✓ | Last update timestamp |
| `description` | string | ○ | Product description |
| `categoryId` | string | ○ | CJ category ID |
| `categoryName` | string | ○ | CJ category name |
| `supplierName` | string | ○ | Supplier name |
| `inventory` | number | ○ | Inventory count |

**Indexes:** `by_cjProductId`, `by_sku`, `by_syncedAt`

---

### Medusa PostgreSQL Product Tables

#### `product` Table (Core)

| Column | Type | Nullable | Description |
|--------|------|----------|-------------|
| `id` | text | NO | Unique product ID |
| `title` | text | NO | Product title |
| `handle` | text | NO | URL-friendly slug |
| `subtitle` | text | YES | Product subtitle |
| `description` | text | YES | Full description |
| `is_giftcard` | boolean | NO | Gift card flag |
| `status` | text | NO | draft/published/rejected |
| `thumbnail` | text | YES | Thumbnail image URL |
| `weight` | text | YES | Product weight |
| `length` | text | YES | Length dimension |
| `height` | text | YES | Height dimension |
| `width` | text | YES | Width dimension |
| `origin_country` | text | YES | Country of origin |
| `hs_code` | text | YES | Harmonized System code |
| `mid_code` | text | YES | Manufacturer ID code |
| `material` | text | YES | Material composition |
| `collection_id` | text | YES | FK to product_collection |
| `type_id` | text | YES | FK to product_type |
| `discountable` | boolean | NO | Can apply discounts |
| `external_id` | text | YES | **← Use for cjProductId** |
| `created_at` | timestamptz | NO | Creation timestamp |
| `updated_at` | timestamptz | NO | Last update timestamp |
| `deleted_at` | timestamptz | YES | Soft delete timestamp |
| `metadata` | jsonb | YES | Custom JSON metadata |

#### `product_variant` Table (SKU Level)

| Column | Type | Nullable | Description |
|--------|------|----------|-------------|
| `id` | text | NO | Variant ID |
| `title` | text | NO | Variant title |
| `sku` | text | YES | **← Maps to cjMyProducts.sku** |
| `barcode` | text | YES | Barcode |
| `ean` | text | YES | European Article Number |
| `upc` | text | YES | Universal Product Code |
| `allow_backorder` | boolean | NO | Backorder allowed |
| `manage_inventory` | boolean | NO | Track inventory |
| `hs_code` | text | YES | HS code |
| `origin_country` | text | YES | Country of origin |
| `mid_code` | text | YES | MID code |
| `material` | text | YES | Material |
| `weight` | integer | YES | Weight (int) |
| `length` | integer | YES | Length |
| `height` | integer | YES | Height |
| `width` | integer | YES | Width |
| `metadata` | jsonb | YES | Custom metadata |
| `variant_rank` | integer | YES | Sort order |
| `product_id` | text | YES | FK to product |
| `thumbnail` | text | YES | Variant image |
| `created_at` | timestamptz | NO | Created |
| `updated_at` | timestamptz | NO | Updated |
| `deleted_at` | timestamptz | YES | Deleted |

#### `image` Table

| Column | Type | Nullable | Description |
|--------|------|----------|-------------|
| `id` | text | NO | Image ID |
| `url` | text | NO | **← Maps to bigImage** |
| `metadata` | jsonb | YES | Image metadata |
| `rank` | integer | NO | Display order |
| `product_id` | text | NO | FK to product |
| `created_at` | timestamptz | NO | Created |
| `updated_at` | timestamptz | NO | Updated |
| `deleted_at` | timestamptz | YES | Deleted |

#### `price` Table (via `product_variant_price_set`)

| Column | Type | Nullable | Description |
|--------|------|----------|-------------|
| `id` | text | NO | Price ID |
| `title` | text | YES | Price name |
| `price_set_id` | text | NO | FK to price_set |
| `currency_code` | text | NO | e.g., "usd" |
| `amount` | numeric | NO | **← Maps to price** |
| `raw_amount` | jsonb | NO | Raw amount data |
| `min_quantity` | numeric | YES | Min qty for this price |
| `max_quantity` | numeric | YES | Max qty for this price |
| `price_list_id` | text | YES | FK to price_list |
| `created_at` | timestamptz | NO | Created |
| `updated_at` | timestamptz | NO | Updated |
| `deleted_at` | timestamptz | YES | Deleted |

#### `product_category` Table

| Column | Type | Nullable | Description |
|--------|------|----------|-------------|
| `id` | text | NO | Category ID |
| `name` | text | NO | **← Maps to categoryName** |
| `description` | text | NO | Category description |
| `handle` | text | NO | URL slug |
| `is_active` | boolean | NO | Active status |
| `is_internal` | boolean | NO | Internal only |
| `parent_category_id` | text | YES | FK for nesting |
| `metadata` | jsonb | YES | Custom metadata |

---

### Schema Mapping Analysis

#### ✅ Direct Mappings (Close Match)

| CJ Field | Medusa Field | Notes |
|----------|--------------|-------|
| `nameEn` | `product.title` | Direct string |
| `sku` | `product_variant.sku` | Direct string |
| `bigImage` | `image.url` | Needs image record creation |
| `price` | `price.amount` | Needs price_set linkage |
| `description` | `product.description` | Optional in CJ |
| `categoryName` | `product_category.name` | Needs category lookup/create |
| `cjProductId` | `product.external_id` | Link back to CJ |
| `cjCreatedAt` | `product.metadata.cj_created_at` | Store in metadata |
| `supplierName` | `product.metadata.supplier` | Store in metadata |

#### ⚠️ Requires Transformation

| CJ Field | Issue | Solution |
|----------|-------|----------|
| `productNames` | Array of names | Store in `metadata`, use first for `subtitle` |
| `productType` | CJ-specific code | Map to `product_type.value` or `metadata` |
| `inventory` | Simple number | Use `inventory_item` module with levels |
| `listedShopNum` | CJ metric | Store in `metadata.cj_shop_count` |

#### ❌ Missing in CJ (Need Defaults)

| Medusa Field | Required | Default Value |
|--------------|----------|---------------|
| `handle` | YES | Generate from `nameEn` (slugify) |
| `status` | YES | `"draft"` |
| `is_giftcard` | YES | `false` |
| `discountable` | YES | `true` |
| `product_variant.title` | YES | Use `nameEn` or "Default" |
| `product_variant.allow_backorder` | YES | `false` |
| `product_variant.manage_inventory` | YES | `true` |

#### 🆕 New Fields for medusaMyProducts

| Field | Purpose |
|-------|---------|
| `medusaProductId` | After sync, store Medusa product.id |
| `syncStatus` | "pending" / "synced" / "failed" |
| `lastMedusaSyncAt` | When last synced to Medusa |
| `modifiedFields` | Track which fields were edited |
| `priceMarkup` | Optional: our markup from CJ price |
| `customTitle` | Override CJ nameEn |
| `customDescription` | Override CJ description |

---

### Related Medusa Tables (Reference)

| Table | Purpose |
|-------|---------|
| `product_collection` | Group products (e.g., "Chandeliers") |
| `product_tag` | Searchable tags |
| `product_type` | Product types |
| `product_option` | Variant options (color, size) |
| `product_option_value` | Option values |
| `product_sales_channel` | Which channels show product |
| `product_shipping_profile` | Shipping profiles |
| `product_variant_inventory_item` | Inventory tracking |


cjMyProducts (raw CJ data)
    ↓ Transform & curate in UI
    ↓ Mark as "ready" when complete
    
Convex Staging Tables (mirrors Medusa schema)
├── medusaProducts         → syncs to → product
├── medusaProductVariants  → syncs to → product_variant
├── medusaImages           → syncs to → image
├── medusaPrices           → syncs to → price
└── medusaCategories       → syncs to → product_category
    
    ↓ When row is "approved/ready"
    
Medusa PostgreSQL (actual commerce data)

# Product Content Generator Agent

The Product Content Generator is a TypeScript CLI agent that uses Claude claude-sonnet-4-20250514 to generate professional e-commerce content for products in the Convex `medusaProducts` table.

### Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│  catalogue/agents/product-content-generator.ts                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  CLI Arguments:                                                 │
│    --limit N       Process N products (default: 10)             │
│    --product-id    Process specific product by Convex _id       │
│    --all           Process all products needing content         │
│    --dry-run       Preview without saving to Convex             │
│    --force         Regenerate content even if already exists    │
│                                                                 │
└──────────────────────────┬──────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│  Anthropic Claude claude-sonnet-4-20250514                                        │
├─────────────────────────────────────────────────────────────────┤
│  System Prompt: E-commerce copywriter for lighting/home decor   │
│                                                                 │
│  Input: Original CJ product data (title, description, images)   │
│                                                                 │
│  Output (JSON):                                                 │
│    • title (max 60 chars) - Professional, keyword-rich          │
│    • subtitle (max 100 chars) - Value proposition               │
│    • description (150-300 words) - Feature-focused, persuasive  │
│    • tags (5-10 strings) - SEO keywords, categories             │
│    • productType - Category classification                      │
│    • suggestedCategories - Medusa category recommendations      │
│    • seoTitle (max 60 chars) - Meta title                       │
│    • seoDescription (max 155 chars) - Meta description          │
│                                                                 │
└──────────────────────────┬──────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│  Convex medusaProducts Table                                    │
├─────────────────────────────────────────────────────────────────┤
│  Direct Field Updates:                                          │
│    • title         ← AI-generated title                         │
│    • subtitle      ← AI-generated subtitle                      │
│    • description   ← AI-generated description                   │
│    • typeValue     ← AI-generated productType                   │
│                                                                 │
│  Metadata Storage (for review):                                 │
│    metadata.aiContent: {                                        │
│      generatedAt: ISO timestamp                                 │
│      model: "claude-sonnet-4-20250514"                            │
│      tags: string[]                                             │
│      suggestedCategories: string[]    ← stored for later review │
│      seoTitle: string                                           │
│      seoDescription: string                                     │
│      originalTitle: string            ← preserved for reference │
│    }                                                            │
└─────────────────────────────────────────────────────────────────┘
```

### Processing Flow

1. **Query**: Fetches products from `medusaProducts` where `metadata.aiContent` is undefined (or all with `--force`)
2. **Sequential Processing**: Processes one product at a time with **500ms delay** between API calls (rate limit protection)
3. **AI Generation**: Sends product context to Claude with structured JSON output schema
4. **Storage**: Writes generated content to Convex fields + stores review data in `metadata.aiContent`
5. **Logging**: Reports success/failure for each product with running totals

### Key Design Decisions

| Decision | Rationale |
|----------|-----------|
| Sequential with 500ms delays | Prevents Anthropic rate limiting, allows graceful interruption |
| Categories in metadata | Enables human review before linking to Medusa categories |
| Original title preserved | Allows comparison and rollback if needed |
| Dry-run mode | Test prompt engineering without writing to database |
| Force flag | Regenerate content after prompt improvements |

### Environment

```bash
# Required in catalogue/.env.local
ANTHROPIC_API_KEY=sk-ant-...
CONVEX_URL=http://127.0.0.1:3210  # or production URL
```

### Usage

```bash
cd catalogue

# Process 10 products (default)
npx ts-node agents/product-content-generator.ts

# Process specific product
npx ts-node agents/product-content-generator.ts --product-id abc123

# Preview without saving
npx ts-node agents/product-content-generator.ts --dry-run --limit 5

# Force regenerate all products
npx ts-node agents/product-content-generator.ts --all --force
```

## Setup

Add your Anthropic API key to `catalogue/.env.local`:

```bash
# Required for AI content generation
ANTHROPIC_API_KEY=sk-ant-api03-...

# Convex URL (should already be set)
NEXT_PUBLIC_CONVEX_URL=http://127.0.0.1:3210
```

## Convex Schema

The agent uses these existing fields in the `medusaProducts` table:

| Field | Type | Purpose |
|-------|------|---------|
| `title` | string | AI-generated product title |
| `subtitle` | string | AI-generated value proposition |
| `description` | string | AI-generated product description |
| `typeValue` | string | Product type classification |
| `handle` | string | URL slug (auto-regenerated from new title) |
| `metadata.aiContent` | object | Review data and generation metadata |

### metadata.aiContent Structure

```json
{
  "generatedAt": "2026-01-16T12:00:00.000Z",
  "model": "claude-sonnet-4-20250514",
  "tags": ["pendant-light", "modern", "dining-room"],
  "suggestedCategories": ["Lighting > Pendant Lights > Modern"],
  "seoTitle": "Modern Glass Pendant Light | Home Decor",
  "seoDescription": "Illuminate your space with our elegant glass pendant...",
  "originalTitle": "2024 New Arrival Lamp Glass Nordic Light..."
}
```

## API Reference

### Queries

```typescript
// Get products needing content
const products = await convex.query(api.medusaStaging.getProductsNeedingContent, {
  limit: 10,
  includeWithContent: false,  // true for --force mode
});

// Get single product by ID
const product = await convex.query(api.medusaStaging.getProductForAiContent, {
  productId: "abc123...",
});

// Get generation statistics
const stats = await convex.query(api.medusaStaging.getAiContentStats, {});
// Returns: { totalProducts, withAiContent, withoutAiContent, percentComplete }
```

### Mutations

```typescript
// Save AI-generated content
await convex.mutation(api.medusaStaging.updateProductAiContent, {
  productId: "abc123...",
  title: "Modern Glass Pendant Light",
  subtitle: "Elegant minimalist design",
  description: "Transform your space...",
  productType: "Pendant Light",
  tags: ["modern", "glass", "pendant"],
  suggestedCategories: ["Lighting > Pendants"],
  seoTitle: "Modern Glass Pendant | Shop",
  seoDescription: "Discover our collection...",
  model: "claude-sonnet-4-20250514",
});
```

# Product Categories

## Product Classification System

Products are organized into **7 main types** with **468 unique subcategories** and an **LED tag** for LED-specific products.

---

### Main Product Types

| Type | Products | LED | Subcategories | Description |
|------|----------|-----|---------------|-------------|
| **Chandeliers** | 307 | 18 | 120 | Hanging multi-light fixtures with decorative arms or branches |
| **Pendants** | 119 | 10 | 64 | Single or multi-light fixtures hanging by cord, chain, or rod |
| **Wall** | 139 | 28 | 88 | Wall-mounted sconces, wall lights, and wall lamps |
| **Ceiling** | 62 | 30 | 40 | Flush mount and semi-flush mount ceiling fixtures |
| **Table & Floor** | 167 | 19 | 101 | Table lamps, desk lamps, and floor lamps |
| **Outdoor** | 23 | 9 | 25 | Outdoor and garden lighting fixtures |
| **Accessories** | 9 | 8 | 30 | LED strips, night lights, smart bulbs, mirrors, specialty lighting |
| **TOTAL** | **826** | **122** | **468** | |

---

### Type Mapping Rules

Original AI-classified types are mapped to the 7 main types:

| Original Type | Count | → Main Type |
|---------------|-------|-------------|
| Chandeliers | 307 | **Chandeliers** |
| Table Lamps | 156 | **Table & Floor** |
| Wall Sconces | 135 | **Wall** |
| Pendant Lights | 119 | **Pendants** |
| Ceiling Lights | 62 | **Ceiling** |
| Outdoor Lighting | 12 | **Outdoor** |
| Outdoor Lights | 11 | **Outdoor** |
| Floor Lamps | 11 | **Table & Floor** |
| Wall Lights | 4 | **Wall** |
| LED Strips | 3 | **Accessories** |
| Night Lights | 2 | **Accessories** |
| Mirrors | 1 | **Accessories** |
| Smart Bulbs | 1 | **Accessories** |
| Specialty Lights | 1 | **Accessories** |
| LED Lights | 1 | **Accessories** |

---

### LED Tag

Products with LED in their category path receive the `LED` tag:
- **122 products** (14.8% of catalog) are tagged as LED
- Highest LED concentration: **Ceiling** (48%), **Accessories** (89%)

---

### Medusa IDs Reference

#### Product Types
```javascript
const PRODUCT_TYPE_IDS = {
  "Chandeliers": "ptyp_01KF7331ET11VZXEDJ16AP9S40",
  "Pendants": "ptyp_01KF7331F53JWM7232WC1GB87S",
  "Wall": "ptyp_01KF7331FBRP67VBZN868BDSRJ",
  "Ceiling": "ptyp_01KF7331FGT6NGBQJ6YYJ3TN20",
  "Table & Floor": "ptyp_01KF7331FNH30XX3P0WW8NAJSP",
  "Outdoor": "ptyp_01KF7331FT3ET088R4KFRASG9X",
  "Accessories": "ptyp_01KF7331FZJJ0ZM6WHWB6GD7T5",
};
```

#### Product Tag
```javascript
const LED_TAG_ID = "ptag_01KF7331G5EQDVAC94SFNADA62";
```

#### Main Categories
```javascript
const CATEGORY_IDS = {
  "Chandeliers": "pcat_01KF736S869NMN0XA35AA07XPM",
  "Pendants": "pcat_01KF73711R8NF7FV7BKB96PWA6",
  "Wall": "pcat_01KF7375B8QDW6HP07AHYCKZQ8",
  "Ceiling": "pcat_01KF737B8B0SPRD4DV9W2RGTM8",
  "Table & Floor": "pcat_01KF737DY59JFQDPA35FTCZ7HM",
  "Outdoor": "pcat_01KF737MPK7JZFATG1DBV0RBC8",
  "Accessories": "pcat_01KF737PCZPCQ39EMRNTJHQT9B",
};
```

---

### Subcategories by Main Type

#### Chandeliers (120 subcategories)
Top styles: Modern (178), Lighting (65), Industrial (48), Nordic Style (35), Contemporary (29), Natural Materials (20), Vintage (17), Glass (17), Rustic (13), Crystal (12)

#### Pendants (64 subcategories)
Top styles: Modern (66), Industrial (24), Nordic Style (19), Lighting (12), Contemporary (12), Nordic (12), Glass (8), Vintage (7), Natural Materials (6)

#### Wall (88 subcategories)
Top styles: Modern (95), Lighting (26), Wall Mounted (23), Industrial (20), Contemporary (13), Nordic Style (12), Vintage (12), Bedside Lamps (9), Bedroom (8)

#### Ceiling (40 subcategories)
Top styles: Modern (46), Lighting (18), Flush Mount (17), Ceiling Fixtures (10), Minimalist (9), Contemporary (8), LED (5), Geometric (4)

#### Table & Floor (101 subcategories)
Top styles: Modern (98), Lighting (79), Bedside (20), Bedside Lamps (17), Table Lamps (16), Contemporary (13), Decorative (11), Desk Lamps (11), Nordic Style (10)

#### Outdoor (25 subcategories)
Top styles: Solar Lights (10), Outdoor (5), Lighting (5), Wall Sconces (5), Pathway Lights (5), Garden Lights (3), Solar LED (3), Wall Lights (3), Step Lights (3)

#### Accessories (30 subcategories)
Top styles: Night Lights (2), Projection (2), Linear (1), Modern (1), Motion Sensor (1), Bathroom Mirrors (1), Smart Lighting (1)

---

### Future Index: By Room *(Alternative Hierarchy)*

> **Note:** This room-based index is planned for future implementation. Products will be cross-referenced to rooms based on their subcategories.

| Room | Suggested Subcategories |
|------|-------------------------|
| **Bedroom** | Bedside Lamps, Ceiling Fixtures, Ceiling Lights, Pendant Lights, Wall Lights, Wall Mounted, Reading Lights |
| **Living Room** | Accent Lighting, Ceiling Fixtures, Chandeliers, Statement Pieces, Wall Lights, Floor Lamps, Table Lamps |
| **Dining Room** | Chandeliers, Contemporary, Japanese Style, LED Fixtures, Linear, Linear Chandeliers, Modern, Nordic Style, Pendant Fixtures, Pendant Lights, Statement Chandeliers |
| **Kitchen** | Island Lights, Island Pendants, Pendant Lights, Chandeliers |
| **Hallway & Corridor** | Contemporary, Wall Mounted, Sconces, Ceiling Lights, Flush Mount |
| **Bathroom** | Vanity Lights, Mirror Lights, Wall Sconces, Modern |
| **Study/Office** | Desk Lamps, Task Lighting, Reading Lamps, Adjustable Lamps |
| **Kids Room** | Chandeliers, Night Lights, Cartoon, Novelty, Decorative |
| **Outdoor** | Balcony, Pathway Lights, Security Lights, Solar Lights, Step Lights, String Lights, Wall Lights, Garden Lights |

---

### Future Index: By Style

> **Note:** This style-based index is planned for cross-referencing products by design aesthetic.

| Style | Description |
|-------|-------------|
| **Modern** | Geometric, Glass, Linear Lights, Luxury, Minimalist, Nordic Style, Scandinavian |
| **Nordic/Scandinavian** | Clean lines, natural materials, functional design |
| **Industrial** | Copper, Vintage & Retro, Steampunk, Wire Cage, Metal |
| **Vintage/Retro** | Automotive, Chandeliers, Industrial Style, Antique finishes |
| **Asian Inspired** | Japanese, Chinese, Oriental Style, Bamboo, Paper |
| **Bohemian/Boho** | Natural Materials, Woven, Macrame, Rattan |
| **Rustic/Farmhouse** | Metal Fixtures, Wood, Antler Fixtures, Natural |
| **Contemporary** | Accent Lamps, Acrylic, Geometric, Globe Lights, Minimalist |
| **Art Deco** | Geometric patterns, luxurious materials, bold styling |

---

### Data Storage

#### Convex (`medusaProducts.metadata.classification`)
```json
{
  "mainType": "Chandeliers",
  "isLED": false,
  "subcategories": ["Modern", "Crystal", "Luxury"],
  "originalCategories": ["Lighting > Chandeliers > Modern", ...],
  "classifiedAt": "2026-01-17T..."
}
```

#### Medusa Structure
- **Product Type**: Links to one of 7 `product_type` records
- **Product Tag**: Links to `LED` tag if `isLED: true`
- **Product Categories**: Links to main category + subcategories

---

### Scripts Reference

| Script | Purpose |
|--------|---------|
| `scripts/classify-product-types.ts` | Preview classification (dry-run) |
| `scripts/update-product-classification.ts` | Apply classification to Convex |
| `scripts/create-medusa-types.ts` | Create 7 types + LED tag in Medusa |
| `scripts/create-medusa-categories.ts` | Create 473 categories in Medusa |
| `scripts/analyze-subcategories.ts` | Analyze subcategory distribution |

# Image Variant Mapping

## Image Hosting & CDN Evaluation (February 2026)

### Current Image Pipeline

```
CJ Dropshipping API → bigImage URL → Convex (medusaImages.url) → Medusa (image table) → Storefront
```

Product images are **hotlinked directly from CJ's servers** at `cf.cjdropshipping.com`. The URL flows through the pipeline untouched:

1. CJ API returns `bigImage` (e.g. `https://cf.cjdropshipping.com/...`)
2. URL string is stored in Convex `medusaImages.url` and `medusaProducts.thumbnail`
3. `push-to-medusa.ts` sends the same URL to Medusa's `image` and `thumbnail` fields
4. Next.js `<Image>` component fetches from `cf.cjdropshipping.com` (whitelisted in `storefront/next.config.js`)

**Key limitation:** CJ provides **one `bigImage` per product** — not per variant. There is no way to map specific images to specific finish/color options using CJ-hosted images alone.

---

### Risks of Hotlinking CJ Images

| Risk | Impact | Severity |
|------|--------|----------|
| **CJ controls uptime** | If CJ's CDN goes down or throttles, entire storefront shows broken images | 🔴 Critical |
| **CJ can change/remove URLs** | Products removed from CJ shelves → dead image links, no fallback | 🔴 Critical |
| **No image transformations** | Cannot crop, resize, or serve WebP/AVIF — serve whatever CJ provides | 🟠 High |
| **No variant-specific images** | Cannot map images to "Gold" vs "White" finishes without self-hosting | 🔴 Critical |
| **Performance variability** | CJ's CDN is optimized for their marketplace, not Elvato's customer geography | 🟡 Medium |
| **No brand control** | Cannot watermark, color-correct, or retouch supplier photos | 🟡 Medium |

---

### ConvexFS + Bunny CDN: Capability Comparison

| Capability | Current (CJ Hotlink) | ConvexFS + Bunny CDN |
|-----------|----------------------|---------------------|
| Image availability | Dependent on CJ | Self-owned — permanent |
| Edge delivery | CJ's CDN (China-optimized) | Bunny's global CDN (190+ PoPs) |
| **Variant-to-image mapping** | ❌ Not possible | ✅ Store a bunny URL per variant/option |
| Image transforms | ❌ None | ✅ Bunny Optimizer: resize, WebP/AVIF, crop on-the-fly |
| Auth/security | Public CJ URLs | Token-signed URLs |
| Path-based organization | N/A | `/products/{pid}/variants/{finish}.webp` |
| Convex integration | N/A | Native — same DB, same auth |
| Cost | Free (hotlinked) | ~$0.01/GB storage + ~$0.01/GB bandwidth |

---

### Why Self-Hosted Images Are Required for Variant Mapping

To show a gold swatch that, when hovered/clicked, displays the gold version of a fixture, the system needs to:

1. **Have multiple images** (one per finish) stored in a location we control
2. **Associate each image URL** with a specific option value in the data model

The existing `medusaImages` table in `convex/schema.ts` already stores image URLs per product. The migration path is to swap CJ URLs for self-hosted bunny CDN URLs and extend the schema to support per-variant image references.

---

### ConvexFS Architecture Fit

ConvexFS is a natural fit for this project because:

- **Already running Convex** for the catalogue — no new infrastructure
- **Path-based file organization** maps directly to product/variant structure
- **bunny.net CDN** handles global edge delivery + on-the-fly image optimization
- **Token authentication** prevents hotlinking of curated images
- **Atomic transactions** for file operations (move/copy/delete without races)
- **File expiration** for automatic cleanup of temporary uploads

#### ConvexFS Setup Requirements

| Variable | Source |
|----------|--------|
| `BUNNY_STORAGE_ZONE` | Name of the bunny.net storage zone |
| `BUNNY_CDN_HOSTNAME` | Full hostname of the CDN pull zone (e.g. `elvato-cdn.b-cdn.net`) |
| `BUNNY_TOKEN_KEY` | Secret key for generating CDN access tokens |
| `BUNNY_API_KEY` | API key for uploading blobs (from FTP & API access → Password) |
| `BUNNY_REGION` | Only needed if main region is non-Frankfurt |

---

### Alternatives Considered

| Solution | Pros | Cons |
|----------|------|------|
| **ConvexFS + bunny.net** | Native Convex integration, global CDN, token auth, affordable | Not free tier, bunny.net dependency |
| **Medusa built-in file service (S3/MinIO)** | Works for basic uploads | No CDN transforms, separate from Convex |
| **Cloudinary / Imgix** | Mature image optimization | Separate billing, not integrated with Convex |
| **bunny.net directly** (no ConvexFS) | Cheaper for CDN-only | Lose Convex-native file management |

---

### Recommended Migration Path

1. **Set up ConvexFS + bunny.net** (~10 min — storage zone, CDN pull zone, env vars)
2. **Write a batch script** to download existing CJ images and upload to ConvexFS
3. **Update `medusaImages.url`** and `medusaProducts.thumbnail` to point to bunny CDN URLs
4. **Add `variantImageUrl` field** to variant data to support per-finish images
5. **Update `storefront/next.config.js`** `remotePatterns` to allow the bunny hostname
6. **Update storefront components** to serve variant-specific images on swatch hover/selection

### Schema Changes Required

```typescript
// In medusaProductVariants — add per-variant image reference
thumbnail: v.optional(v.string()),       // Already exists — update to bunny URL
variantImageUrl: v.optional(v.string()), // New: primary image for this specific variant

// In medusaImages — add variant association
medusaVariantId: v.optional(v.id("medusaProductVariants")), // New: link image to specific variant
```

### Estimated Image Budget

| Metric | Count |
|--------|-------|
| Products | 826 |
| Physical variants requiring unique images | 3,019 |
| Current CJ-hosted images | 6,937 |
| Missing variant images | 437 |
| **Estimated storage** (avg 500KB/image) | **~3.4 GB** |
| **Estimated monthly CDN cost** (10K visits/mo) | **< $1/month** |

## Overview

The Image Variant Mapping system categorizes product variants into **physical** (requiring unique images) vs. **non-physical** (sharing images). This reduces the total image requirements by identifying which variants can share the same product photography.

### Option Classification

| **Physical Options** (Require Unique Images) | **Non-Physical Options** (Share Images) |
|---------------------------------------------|----------------------------------------|
| Finish/Color (Black, Gold, White, etc.) | Size (when shape is same) |
| Number of Lights (1-head, 3-head, 5-head) | Color Temperature (3000K, 4000K, etc.) |
| | Wattage (5W, 10W, 15W, etc.) |
| | Voltage (110V, 220V) |
| | Dimmable (Yes/No) |
| | Bulb Type (E26, E27, G9, etc.) |
| | Material |
| | Style |
| | Cord Length |

### Analysis Results (January 2026)

| Metric | Value |
|--------|-------|
| **Products Analyzed** | 826 |
| **Total Variants** | 11,492 |
| **Physical Variants** (need unique images) | 3,019 |
| **Non-Physical Variants** (share images) | 8,473 |
| **Image Requirement Reduction** | **73.7%** |

### Image Coverage Status

| Status | Products | Description |
|--------|----------|-------------|
| ✅ Complete | 740 (89.6%) | All physical variants have images |
| ⚠️ Partial | 86 (10.4%) | Some physical variants missing images |
| ❌ Missing | 0 (0%) | No images assigned |

### Current vs Required Images

| Metric | Count |
|--------|-------|
| Required Images | 3,019 |
| Current Images | 6,937 |
| Missing Images | 437 |
| Average Coverage | 96.4% |

### Priority Breakdown (86 Products Needing Attention)

| Priority | Missing Images | Products | Est. Effort |
|----------|---------------|----------|-------------|
| 🔴 Critical | 10+ images | 9 | High |
| 🟠 High | 5-9 images | 7 | Medium-High |
| 🟡 Medium | 2-4 images | 47 | Medium |
| 🟢 Low | 1 image | 23 | Low |

### Key Insight

> **73.7% reduction**: Instead of needing 11,492 unique images (one per variant), we only need **3,019 unique images** by grouping variants that share the same physical appearance.

### Data Location

The variant mapping data is stored in Convex table `variantMapping` with the following structure:
- `physicalVariantGroups` - Groups of variants sharing the same image
- `physicalOptions` - Options that change appearance (Finish, Number of Lights)
- `nonPhysicalOptions` - Options that don't change appearance (Size, Wattage, etc.)

### Related Files

| File | Purpose |
|------|---------|
| `convex/schema.ts` | variantMapping table definition |
| `convex/variantMapping.ts` | Queries and mutations for variant analysis |
| `scripts/analyze-variant-images.ts` | Script to populate variantMapping table |
| `scripts/export-image-report.ts` | Export products needing images |
| `.agents/product-images/needsImage.md` | Executive report of products needing attention |
| `.agents/product-images/assignmentWorkflow.md` | Image assignment workflow guide |

### updated image/ pagination architecture

## ConvexFS + Bunny CDN Implementation (February 2026)

### Architecture Change Summary

Replaced direct CJ Dropshipping image hotlinking with a self-hosted CDN pipeline using **ConvexFS** (Convex component) backed by **Bunny.net Edge Storage + CDN**. Additionally, rewrote the storefront product listing query to use **server-side pagination** instead of client-side fetch-all-then-sort, reducing store page load from **7.8s → ~100ms** (70x improvement).

```
BEFORE:
CJ API → bigImage URL → Medusa DB → Storefront <Image> → cf.cjdropshipping.com (slow, unreliable)
Medusa Store API → 4.3MB response (100 products, full variants+images+metadata) → client-side sort → 7.8s page load

AFTER:
CJ Images → ConvexFS ingest → Bunny.net Edge Storage (NY) → Bunny CDN (190+ PoPs) → Storefront <Image>
Medusa Store API → ~50KB response (12 products, slim fields) → server-side sort → ~100ms page load
```

---

### ConvexFS Setup

#### Component Registration

```typescript
// convex/convex.config.ts
import { defineApp } from "convex/server";
import fs from "convex-fs/convex.config.js";

const app = defineApp();
app.use(fs);

export default app;
```

#### ConvexFS Instance

```typescript
// convex/fs.ts
import { ConvexFS } from "convex-fs";
import { components } from "./_generated/api";

export const fs = new ConvexFS(components.fs, {
  storage: {
    type: "bunny",
    apiKey: process.env.BUNNY_API_KEY!,
    storageZoneName: process.env.BUNNY_STORAGE_ZONE!,
    region: process.env.BUNNY_REGION,          // "ny"
    cdnHostname: process.env.BUNNY_CDN_HOSTNAME!,
    tokenKey: process.env.BUNNY_TOKEN_KEY,     // For signed URLs
  },
});
```

#### HTTP Routes

```typescript
// convex/http.ts
registerRoutes(http, components.fs, fs, {
  pathPrefix: "/fs",
  uploadAuth: async (ctx) => {
    const identity = await ctx.auth.getUserIdentity();
    return identity !== null;  // Authenticated uploads only
  },
  downloadAuth: async () => true,  // Public downloads (product images)
});
```

Downloads return a **302 redirect** to a signed Bunny CDN URL — the browser fetches the image directly from the nearest edge PoP.

---

### Environment Variables

#### Convex Dashboard (Bunny.net)

| Variable | Value | Source |
|----------|-------|--------|
| `BUNNY_API_KEY` | `05ee2f91-...` | Bunny dashboard → FTP & API Access → Password |
| `BUNNY_STORAGE_ZONE` | `elvatostorage` | Must be **lowercase** (not `elvatoStorage`) |
| `BUNNY_REGION` | `ny` | Required for non-Frankfurt storage zones |
| `BUNNY_CDN_HOSTNAME` | `elvatoStorage-CDN.b-cdn.net` | Pull Zone hostname from Bunny dashboard |
| `BUNNY_TOKEN_KEY` | `d7b4b81b-...` | Pull Zone → Security → Token Authentication Key |

#### Convex Dashboard (Medusa — for ingestion action)

| Variable | Value |
|----------|-------|
| `MEDUSA_BACKEND_URL` | `http://localhost:9000` |
| `MEDUSA_PUBLISHABLE_KEY` | `pk_42863ea1...` |

#### Storefront `.env.local`

| Variable | Value |
|----------|-------|
| `NEXT_PUBLIC_CONVEX_URL` | `http://127.0.0.1:3210` |
| `NEXT_PUBLIC_CONVEX_SITE_URL` | `http://127.0.0.1:3211` |

#### Next.js Image Domains (`storefront/next.config.js`)

```javascript
images: {
  remotePatterns: [
    { protocol: "https", hostname: "elvatoStorage-CDN.b-cdn.net" },  // Bunny CDN
    { protocol: "http", hostname: "127.0.0.1", port: "3211" },       // Convex site (local dev)
    { protocol: "https", hostname: "cf.cjdropshipping.com" },        // Fallback
  ],
}
```

---

### ⚠️ Bunny.net Auth Troubleshooting

Issues encountered and resolved during setup:

| Issue | Symptom | Fix |
|-------|---------|-----|
| Storage zone casing | 401 Unauthorized on upload | Use `elvatostorage` (lowercase), not `elvatoStorage` |
| Missing region | 401 Unauthorized on upload | Set `BUNNY_REGION=ny` — required for non-Frankfurt zones |
| CDN hostname format | 404 on download | Use full hostname `elvatoStorage-CDN.b-cdn.net` |
| Token auth | 403 on direct CDN URL | ConvexFS handles token signing automatically via `buildDownloadUrl()` |

---

### File Organization & Path Convention

```
/products/
├── {productHandle}/
│   ├── thumbnail.{ext}           ← Product thumbnail (1 per product)
│   └── images/
│       ├── 0.{ext}               ← Gallery image rank 0
│       ├── 1.{ext}               ← Gallery image rank 1
│       └── ...
```

Extension (`ext`) is determined at ingest time from the `content-type` header: `jpg`, `png`, `webp`, or `gif`.

---

### Convex Backend API (`convex/files.ts`)

#### Queries

| Function | Args | Description |
|----------|------|-------------|
| `listFiles` | `prefix?, paginationOpts` | List all files under a path prefix (paginated) |
| `getFile` | `path` | Get metadata for a single file |
| `getFileUrl` | `path` | Get a signed CDN download URL for one file |
| `getFileUrls` | `paths[]` | Batch-resolve CDN URLs for multiple paths |
| `getProductImages` | `productHandle` | List all images for a product with CDN URLs |
| `getBatchThumbnails` | `handles[]` | **Batch-resolve** CDN thumbnail URLs for multiple products in one query |

#### Mutations

| Function | Args | Description |
|----------|------|-------------|
| `commitFile` | `path, blobId` | Commit an uploaded blob to a path |
| `commitFiles` | `files[]` | Commit multiple blobs atomically |
| `deleteFile` | `path` | Delete a file by path |
| `moveFile` | `sourcePath, destPath` | Move/rename a file |

#### Actions (Image Ingestion)

| Function | Args | Description |
|----------|------|-------------|
| `ingestImage` | `sourceUrl, destPath` | Download a single external image and write to ConvexFS |
| `ingestProductImages` | `handle, thumbnail?, images[]` | Ingest thumbnail + gallery for one product |
| `ingestProductById` | `productId` | *(internal)* Read product from DB and ingest all images |
| `ingestPublishedProductImages` | *(none)* | **Batch ingest** — queries Medusa for published products, cross-references Convex, schedules ingestion for each |

#### Internal Queries

| Function | Args | Description |
|----------|------|-------------|
| `getProductForIngestion` | `productId` | Read product handle + thumbnail + medusaImages rows |
| `getAllProductIds` | *(none)* | Get all `medusaProducts` with `syncStatus: "synced"` |

---

### Image Ingestion Pipeline

#### Batch Ingestion Flow

```
ingestPublishedProductImages (action)
    │
    ├── 1. GET /store/products?limit=100&fields=handle,thumbnail  (Medusa Store API)
    │      → Returns published product handles
    │
    ├── 2. getAllProductIds (internal query)
    │      → Returns synced medusaProducts from Convex
    │
    ├── 3. Cross-reference: match published handles → Convex IDs
    │      → Skip products not in Convex
    │
    └── 4. For each matched product:
           ctx.scheduler.runAfter(0, ingestProductById, { productId })
               │
               ├── getProductForIngestion (internal query)
               │      → Read handle, thumbnail URL, medusaImages rows
               │
               ├── fetch(thumbnail URL) → fs.writeFile(/products/{handle}/thumbnail.{ext})
               │
               └── for each image:
                   fetch(image URL) → fs.writeFile(/products/{handle}/images/{rank}.{ext})
```

#### Ingestion Stats (Initial Run)

| Metric | Value |
|--------|-------|
| Published products (Medusa) | 39 |
| Matched in Convex | 37 |
| Skipped (not in Convex) | 2 |
| Total files ingested | 330 |
| Total data uploaded | 51.9 MB |
| Average file size | ~157 KB |

---

### Storefront Integration

#### CDN Image Utility (`storefront/src/lib/data/convex-images.ts`)

The storefront queries the Convex backend via HTTP POST to `/api/query` to resolve CDN URLs.

| Function | Purpose |
|----------|---------|
| `fetchProductImages(handle)` | Query Convex for all images of a product; caches in-memory |
| `prefetchThumbnails(handles[])` | **Batch** thumbnail lookup via `getBatchThumbnails` query — one HTTP call for all products on a page |
| `getCdnThumbnail(handle)` | Single thumbnail lookup (checks `thumbCache` first) |
| `getCdnGalleryImages(handle)` | Gallery images sorted by rank, returns `{ url, id }[]` |
| `withCdnImages(product)` | Replace a Medusa product's `thumbnail` and `images` with CDN equivalents; falls back to originals |

#### Caching Strategy

```
Server Process Memory:
├── imageCache: Map<handle, ConvexImage[]>    ← Full product images (1hr revalidate)
└── thumbCache: Map<handle, string | null>    ← Thumbnail URLs (populated by prefetch)
```

- `prefetchThumbnails` is called once per page render with all visible handles → fills `thumbCache`
- Individual `getCdnThumbnail` calls then resolve from cache (zero network overhead)
- Convex queries use `next: { revalidate: 3600 }` for ISR caching

#### N+1 Query Optimization

**Problem:** Each `ProductPreview` component called `getCdnThumbnail` individually → N Convex HTTP calls per page.

**Solution:** Added `getBatchThumbnails` query (Convex) + `prefetchThumbnails` utility (storefront). Pages call `prefetchThumbnails(handles)` before rendering product cards.

| Component | Integration |
|-----------|-------------|
| `product-rail/index.tsx` | Calls `prefetchThumbnails` before rendering featured product cards |
| `paginated-products.tsx` | Calls `prefetchThumbnails` before rendering store page product grid |
| `related-products/index.tsx` | Calls `prefetchThumbnails` before rendering related product cards |
| `product-preview/index.tsx` | Calls `getCdnThumbnail(handle)` → resolves from pre-warmed `thumbCache` |
| `products/[handle]/page.tsx` | Calls `withCdnImages(product)` to replace thumbnail + gallery images with CDN equivalents |

#### Image Request Flow (Storefront)

```
Next.js Server Render
    │
    ├── prefetchThumbnails(["handle-a", "handle-b", ...])
    │       POST → http://127.0.0.1:3210/api/query
    │       body: { path: "files:getBatchThumbnails", args: { handles: [...] } }
    │       → fills thumbCache
    │
    ├── <ProductPreview> renders
    │       getCdnThumbnail("handle-a") → thumbCache hit → CDN URL
    │
    └── Browser loads <Image src="http://127.0.0.1:3211/fs/blobs/...">
            302 redirect → https://elvatoStorage-CDN.b-cdn.net/...?token=...&expires=...
            → Edge-delivered image from nearest Bunny PoP
```

---

### Server-Side Pagination Fix (`storefront/src/lib/data/products.ts`)

#### Problem

The `listProductsWithSort` function fetched **all products** (`limit: 100`) with heavy field expansion (`*variants.images,+metadata,+tags`) on every store page load, resulting in:

| Metric | Before |
|--------|--------|
| Medusa API response size | **4.3 MB** |
| Next.js data cache error | `items over 2MB can not be cached (4316008 bytes)` |
| Store page load time | **7.8 seconds** |
| Approach | Client-side: fetch all → sort → slice for current page |

#### Solution

Two changes applied:

**1. Server-side pagination for `created_at` sort (default)**

Instead of fetching all 100 products and sorting/paginating in JavaScript, the function now passes `limit`, `offset`, and `order: "-created_at"` directly to Medusa's Store API, which returns only 12 products per page:

```typescript
// For created_at (default), use server-side sorting + pagination
return listProducts({
  pageParam: page,
  queryParams: {
    ...queryParams,
    limit,
    order: "-created_at",
    fields: LISTING_FIELDS,
  },
  countryCode,
})
```

**2. Slim fields constant for all listing queries**

Since images are now served from Bunny CDN via ConvexFS, listing queries no longer need `*variants.images`, `+metadata`, or `+tags`:

```typescript
// Before (4.3MB response):
fields: "*variants.calculated_price,+variants.inventory_quantity,*variants.images,+metadata,+tags,"

// After (~50KB response):
const LISTING_FIELDS = "*variants.calculated_price,+variants.inventory_quantity"
```

**3. Price sort still uses client-side approach** (Medusa doesn't support `order` by `calculated_price`), but with slim fields the response drops from 4.3MB to ~300KB — well within the 2MB cache limit.

#### Results

| Metric | Before | After | Improvement |
|--------|--------|-------|-------------|
| Medusa API response | 4.3 MB | ~50 KB | **98.8% reduction** |
| Store page load time | 7.8s | ~100ms | **70x faster** |
| Next.js cache | ❌ Exceeds 2MB limit | ✅ Cacheable | Fixed |
| Products fetched per page | 100 (all) | 12 (current page) | **88% reduction** |
| Approach | Client-side sort + slice | Server-side sort + paginate | Proper |

---

### File Reference

| File | Purpose |
|------|---------|
| `convex/convex.config.ts` | Registers ConvexFS component via `app.use(fs)` |
| `convex/fs.ts` | ConvexFS instance with Bunny.net storage configuration |
| `convex/http.ts` | HTTP routes at `/fs` prefix (upload auth, public downloads) |
| `convex/files.ts` | All queries, mutations, and ingestion actions |
| `storefront/src/lib/data/convex-images.ts` | CDN image utilities: fetch, prefetch, batch, replace |
| `storefront/src/lib/data/products.ts` | `listProductsWithSort` — server-side pagination + slim fields |
| `storefront/src/modules/products/components/product-preview/index.tsx` | CDN thumbnail rendering with fallback |
| `storefront/src/app/[countryCode]/(main)/products/[handle]/page.tsx` | `withCdnImages` for product detail page |
| `storefront/src/modules/home/components/featured-products/product-rail/index.tsx` | `prefetchThumbnails` for home page |
| `storefront/src/modules/store/templates/paginated-products.tsx` | `prefetchThumbnails` for store listing |
| `storefront/src/modules/products/components/related-products/index.tsx` | `prefetchThumbnails` for related products |
| `storefront/next.config.js` | Bunny CDN + Convex site URL in `remotePatterns` |

---

### Next Steps

| Task | Status | Description |
|------|--------|-------------|
| ConvexFS setup | ✅ Done | Component registered, Bunny.net configured |
| Image ingestion | ✅ Done | 37 products, 330 files, 51.9 MB |
| Storefront CDN integration | ✅ Done | Thumbnails + gallery images from CDN |
| N+1 query optimization | ✅ Done | Batch thumbnail prefetch |
| Server-side pagination | ✅ Done | 7.8s → 100ms store page load |
| Variant-specific images | ⬜ Planned | Map images to finish/color options per variant |
| Production Bunny CDN | ⬜ Planned | Update env vars for production Convex deployment |
| Image optimization | ⬜ Planned | Enable Bunny Optimizer for on-the-fly WebP/AVIF conversion |
| Remaining product ingestion | ⬜ Planned | Ingest images for all 826 products (currently 37 published) |

# Pricing 

continue..... 

we have medusaProductVariants, that are linked to medusaProducts, which are linked to cjMyProducts Id- which lists the corresponding sku number. 

so what we need to do is map the pricing for variants- because similar to images, there may be overlap in pricing where some variants don't require a change in price. 

then, we need to list the individual variants in the convex database for their associated pricing. we'll list the general product and variant information, cj sku, elvato sku, item variant price, all shipping prices (one column per shipping option with associated price), then cj's suggested retail price, then the actual price we'll use to list the variant on our store. 